# Homework 3: Building an NDArray library

# 作业 3：构建 NDArray 库

In this homework, you will build a simple backing library for the processing that underlies most deep learning systems: the n-dimensional array (a.k.a. the NDArray).  Up until now, you have largely been using numpy for this purpose, but this homework will walk you through developing what amounts to your own (albeit much more limited) variant of numpy, which will support both CPU and GPU backends.  What's more, unlike numpy (and even variants like PyTorch), you won't simply call out to existing highly-optimized variants of matrix multiplication or other manipulation code, but actually write your own versions that are reasonably competitive will the highly optimized code backing these standard libraries (by some measure, i.e., "only 2-3x slower" ... which is a whole lot better than naive code that can easily be 100x slower).  This class will ultimately be integrated into `needle`, but for this assignment you can _only_ focus on the ndarray module, as this will be the only subject of the tests.

在本次作业中，你将为大多数深度学习系统底层处理所依赖的核心结构构建一个简单的后端库：n 维数组（也称为 NDArray）。到目前为止，你主要使用 numpy 来完成这类工作，但本作业会引导你开发一个属于自己的 numpy 变体（虽然功能有限得多），并同时支持 CPU 和 GPU 后端。更重要的是，与 numpy（甚至 PyTorch 这类变体）不同，你不会只是调用已有的高度优化矩阵乘法或其他数组操作代码，而是会实际编写自己的版本，使其在一定程度上能够与这些标准库背后的高度优化代码相竞争，例如“只慢 2-3 倍”，这已经比很容易慢 100 倍的朴素代码好得多。这个类最终会被集成进 `needle`，但在本作业中你可以 _只_ 关注 ndarray 模块，因为测试也只会围绕这一部分展开。

**Note**: To avoid exhausting limited GPU resources in Colab, start by using CPU runtime for coding and testing non-GPU functions. Switch to GPU runtime when testing CUDA or GPU-accelerated code. This approach ensures efficient GPU usage and prevents running out of resources during critical tasks.

**注意**：为了避免耗尽 Colab 中有限的 GPU 资源，请先使用 CPU 运行时来编写和测试非 GPU 函数。只有在测试 CUDA 或 GPU 加速代码时再切换到 GPU 运行时。这样可以更高效地使用 GPU，并避免在关键任务期间资源耗尽。


In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw3.git
%cd /content/drive/MyDrive/10714/hw3

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW3_NAME = "hw3"

In [ ]:
!make

The make command reads the Makefile in the current directory. The Makefile contains rules that define how to build targets (like executables or libraries). For each target specified in the Makefile, make checks the timestamps of the target file and its dependencies (like .c, .cpp, or .h files). If any dependency has been modified recently, it must rebuild the target.

`make` 命令会读取当前目录中的 Makefile。Makefile 包含一些规则，用来定义如何构建目标（例如可执行文件或库）。对于 Makefile 中指定的每个目标，`make` 会检查目标文件及其依赖项（如 `.c`、`.cpp` 或 `.h` 文件）的时间戳。如果任何依赖项最近被修改过，它就必须重新构建该目标。


In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

## Getting familiar with the NDArray class

## 熟悉 NDArray 类

As you get started with this homework, you should first familiarize yourself with the `NDArray.py` class we have provided as a starting point for the assignment.  The code is fairly brief (it's ~500 lines, but a lot of these are comments provided for the functions you'll implement).

在开始本次作业时，你应该先熟悉我们提供的 `NDArray.py` 类，它是本作业的起点。这份代码相当简短（大约 500 行，其中很多都是为你将要实现的函数提供的注释）。

At its core, the NDArray class is a Python wrapper for handling operations on generic n-dimensional arrays.  Recall that virtually any such array will be stored internally as a vector of floating point values, i.e.,

从本质上说，NDArray 类是一个 Python 包装器，用来处理通用 n 维数组上的操作。回忆一下，几乎所有这样的数组在内部都会存储为一个浮点数向量，也就是：

```c++
float data[size];
```

and then the actual access to different dimensions of the array are all handled by additional fields (such as the array shape, strides, etc) that indicates how this "flat" array maps to n-dimensional structure.  In order to achieve any sort of reasonable speed, the "raw" operations (like adding, binary operations, but also more structured operations like matrix multiplication, etc), all need to be written at some level in some native language like C++ (including e.g., making CUDA calls).  But a large number of operations likes transposing, broadcasting, sub-setting of matrices, and other, can all be handled by just adjusting the high-level structure of the array, like it's strides.

随后，对数组不同维度的实际访问都会通过额外字段来处理（例如数组形状、步幅等），这些字段说明这个“扁平”的数组如何映射到 n 维结构。为了获得任何合理的速度，“原始”操作（例如加法、二元运算，以及矩阵乘法等更结构化的操作）都需要在某种本地语言层面编写，例如 C++（也包括进行 CUDA 调用）。不过，大量操作，例如转置、广播、矩阵子集选取等，都可以只通过调整数组的高层结构来处理，例如调整它的步幅。

The philosophy behind the NDArray class is that we want _all_ the logic for handling this structure of the array to be written in Python.  Only the "true" low level code that actually performs the raw underlying operations on the flat vector (as well as the code to manage these flat vectors, as they might need to e.g., be allocated on GPUs), is written in C++.  The precise nature of this separation will likely start to make more sense to you as you work through the assignment, but generally speaking everything that can be done in Python, is done in Python; often e.g., at the cost of some inefficiencies ... we call `.compact()` (which copies memory) liberally in order to make the underlying C++ implementations simpler.

NDArray 类背后的设计理念是，我们希望 _所有_ 处理数组结构的逻辑都用 Python 编写。只有那些“真正”的底层代码，也就是实际在扁平向量上执行原始底层操作的代码（以及管理这些扁平向量的代码，因为它们可能需要分配在 GPU 上），才用 C++ 编写。随着你完成作业，这种分离的具体含义会逐渐变得更清楚；但一般来说，凡是能在 Python 中完成的事情，就都在 Python 中完成。这样有时会牺牲一些效率，例如我们会比较频繁地调用 `.compact()`（它会复制内存），以便让底层 C++ 实现更简单。

In more detail, there are five fields within the NDArray class that you'll need to be familiar with (note that the real class member these all these fields is preceded by an underscore, e.g., `_handle`, `_strides`, etc, some of which are then exposed as a public property ... for all your code it's fine to use the internal, underscored version).

更具体地说，NDArray 类中有五个你需要熟悉的字段（注意，这些字段对应的真实类成员名前面都有下划线，例如 `_handle`、`_strides` 等，其中有些随后会作为公开属性暴露出来。对于你编写的代码，直接使用带下划线的内部版本是可以的）。

1. `device` - A object of type `BackendDevice`, which is a simple wrapper that contains a link to the underlying device backend (e.g., CPU or CUDA).

   `device` - 一个 `BackendDevice` 类型的对象，它是一个简单包装器，包含指向底层设备后端（例如 CPU 或 CUDA）的链接。

2. `handle` - A class objected that stores the underlying memory of the array.  This is allocated as a class of type `device.Array()`, though this allocation all happens in the provided code (specifically the `NDArray.make` function), and you don't need to worry about calling it yourself.

   `handle` - 一个类对象，用来存储数组的底层内存。它会被分配为 `device.Array()` 类型的对象，不过这个分配过程都发生在提供的代码中（具体来说是 `NDArray.make` 函数），你不需要担心自己调用它。

3. `shape` - A tuple specifying the size of each dimension in the array.

   `shape` - 一个元组，指定数组每个维度的大小。

4. `strides` - A tuple specifying the strides of each dimension in the array.

   `strides` - 一个元组，指定数组每个维度的步幅。

5. `offset` - An integer indicating where in the underlying `device.Array` memory the array actually starts (it's convenient to store this so we can more easily manage pointing back to existing memory, without having to track allocations).

   `offset` - 一个整数，表示该数组在底层 `device.Array` 内存中实际开始的位置（保存这个值很方便，因为这样我们可以更容易地指回已有内存，而不需要追踪分配过程）。

By manipulating these fields, even pure Python code can perform a lot of the needed operations on the array, such as permuting the dimensions (i.e., transposing), broadcasting, and more.  And then for the raw array manipulation calls, the `device` class has a number of methods (implemented in C++) that contains the necessary implementations.

通过操作这些字段，即使是纯 Python 代码也可以完成许多数组所需的操作，例如维度置换（也就是转置）、广播等。至于原始数组操作调用，`device` 类中有若干方法（用 C++ 实现），其中包含必要的底层实现。

There are a few points to note:

有几点需要注意：

* Internally, the class can use _any_ efficient means of operating on arrays of data as a "device" backend.  Even, for example, a numpy array, but where instead of actually using the `numpy.ndarray` to represent the n-dimensional array, we just represent a "flat" 1D array in numpy, then call the relevant numpy methods to implement all the needed operators on this raw memory.  This is precisely what we do in the `ndarray_backend_numpy.py` file, which essentially provided a "stub reference" that just uses numpy for everything.  You can use this class to help you  better debug your own "real" implementations for the "native" CPU and GPU backends.

  在内部，这个类可以使用 _任何_ 高效的数据数组操作方式作为“设备”后端。甚至可以使用 numpy 数组；但我们并不是实际使用 `numpy.ndarray` 来表示 n 维数组，而是在 numpy 中只表示一个“扁平”的一维数组，然后调用相关的 numpy 方法，在这块原始内存上实现所有需要的算子。这正是我们在 `ndarray_backend_numpy.py` 文件中所做的事情：它本质上提供了一个“桩参考实现”，所有事情都直接使用 numpy。你可以使用这个类来更好地调试自己为“原生”CPU 和 GPU 后端编写的“真实”实现。

* Of particular importance for many of your Python implementations will be the `NDArray.make` call:

  对你的许多 Python 实现来说，尤其重要的是 `NDArray.make` 调用：

```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```

which creates a new NDArray with the given shape, strides, device, handle, and offset.  If `handle` is not specified (i.e., no pre-existing memory is referenced), then the call will allocate the needed memory, but if handle _is_ specified then no new memory is allocated, but the new NDArray points the same memory as the old one.  It is important to efficient implementations that as many of your functions as possible _don't_ allocate new memory, so you will want to use this call in many cases to accomplish this.

它会使用给定的 shape、strides、device、handle 和 offset 创建一个新的 NDArray。如果没有指定 `handle`（也就是没有引用预先存在的内存），那么该调用会分配所需内存；但如果指定了 handle，则不会分配新内存，而是让新的 NDArray 指向与旧数组相同的内存。对于高效实现而言，尽可能让你的函数 _不要_ 分配新内存非常重要，所以你会在很多情况下使用这个调用来实现这一点。

* The NDArray has a `.numpy()` call that converts the array to numpy.  This is _not_ the same as the "numpy_device" backend: this creates an actual `numpy.ndarray` that is equivalent to the given NDArray, i.e., the same dimensions, shape, etc, though not necessarily the same strides (Pybind11 will reallocate memory for matrices that are returned in this manner, which can change the striding).

  NDArray 有一个 `.numpy()` 调用，可以把数组转换为 numpy。这与 “numpy_device” 后端 _不同_：它会创建一个真正的 `numpy.ndarray`，该数组与给定 NDArray 等价，也就是具有相同的维度、形状等，不过步幅不一定相同（Pybind11 会为以这种方式返回的矩阵重新分配内存，这可能会改变步幅）。


## Part 1: Python array operations

## 第 1 部分：Python 数组操作

As a starting point for your class, implement the following functions in the `ndarray.py` file:

作为本课程实现的起点，请在 `ndarray.py` 文件中实现以下函数：

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

The inputs/outputs of these functions are all described in the docstring of the function stub.  It's important to emphasize that _none_ of these functions should reallocate memory, but should instead return NDArrays that share the same memory with `self`, and just use clever manipulation of shape/strides/etc in order to obtain the necessary transformations.

这些函数的输入和输出都在函数桩的 docstring 中描述。需要特别强调的是，这些函数 _都不应该_ 重新分配内存，而应该返回与 `self` 共享同一块内存的 NDArray，并通过巧妙地操作 shape、strides 等字段来获得所需的变换。

To get started you can refer to the hints mentioned in the class slides for transpose, broadcast and slicing operator.

开始时，你可以参考课堂 slides 中关于转置、广播和切片操作符的提示。

One thing to note is that the `__getitem__()` call, unlike numpy, will never change the number of dimensions in the array.  So e.g., for a 2D NDArray `A`, `A[2,2]` would return a still-2D with one row and one column.  And e.g. `A[:4,2]` would return a 2D NDarray with 4 rows and 1 column.

需要注意的一点是，与 numpy 不同，`__getitem__()` 调用永远不会改变数组的维度数量。因此，例如对于一个二维 NDArray `A`，`A[2,2]` 仍会返回一个二维数组，只是它有一行一列。再比如，`A[:4,2]` 会返回一个二维 NDArray，包含 4 行和 1 列。

You can rely on the `ndarray_backend_numpy.py` module for all the code in this section.  You can also look at the results of equivalent numpy operations (the test cases should illustrate what these are).

本节中的所有代码都可以依赖 `ndarray_backend_numpy.py` 模块。你也可以查看等价 numpy 操作的结果（测试用例应该会展示这些结果是什么）。

After implementing these functions, you should pass/submit the following tests.  Note that we test all of these four functions within the test below, and you can incrementally try to pass more tests as you implement each additional function.

实现这些函数后，你应该能够通过并提交下面的测试。注意，我们会在下面的测试中同时测试这四个函数；你也可以在每实现一个额外函数后，逐步尝试通过更多测试。


In [ ]:
!python3 -m pytest -v -k "(permute or reshape or broadcast or getitem) and cpu and not compact"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_python_ops"

## Part 2: CPU Backend - Compact and setitem

## 第 2 部分：CPU 后端 - Compact 和 setitem

Implement the following functions in `ndarray_backend_cpu.cc`:

在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

To see why these are all in the same category, let's consider the implementation of the `Compact()` function.  Recall that a matrix is considered compact if it is layed out sequentially in memory in "row-major" form (but really a generalization of row-many to higher dimensional arrays), i.e. with the last dimension first, followed by the second to last dimension, etc, all the way to the first.  In our implementation, we also require that the total size of allocated backend array be equal to the size of the array (i.e., the underlying array also can't have any data before or after the array data, which e.g., implies that the `.offset` field equals zero).

为了理解为什么这些函数属于同一类，我们先考虑 `Compact()` 函数的实现。回忆一下，如果一个矩阵在内存中按“行主序”形式顺序排布，就认为它是 compact 的（更准确地说，这是把行主序推广到更高维数组）：也就是最后一个维度最先连续变化，然后是倒数第二个维度，依此类推直到第一个维度。在我们的实现中，还要求已分配的后端数组总大小等于该数组的大小（也就是说，底层数组在数组数据之前或之后都不能有额外数据，这也意味着 `.offset` 字段等于零）。

Now let's consider, using a 3D array as a an example, of how a compact call might work.  Here `shape` and `strides` are the shape and strides of the matrix being compacted (i.e., before we have compacted it).

现在我们用一个三维数组作为例子，看看 compact 调用可能如何工作。这里的 `shape` 和 `strides` 是被 compact 的矩阵的形状和步幅（也就是 compact 之前的形状和步幅）。

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[strides[0]*i + strides[1]*j + strides[2]*k];
```

In other words, we're converting from a stride-based representation of the matrix to a purely sequential one.

换句话说，我们正在把矩阵从基于步幅的表示转换为纯顺序表示。

Now, the challenge in implementing `Compact()` is that you want the method to work for any number of input dimensions.  It's easy to specialize for different fixed-dimension-size arrays, but for a generic implementation, you'll want to think about how to do this same operation where you effectively want a "variable number of for loops".  As a hint, one way to do this is to maintain a vector of indices (of size equal to the number of dimensions), and then manually increment them in a loop (including a "carry" operation when any of the reaches their maximum size).

实现 `Compact()` 的挑战在于，你希望这个方法适用于任意数量的输入维度。针对不同固定维度大小的数组进行专门化很容易，但对于通用实现，你需要思考如何完成同样的操作，因为你实际上需要“可变数量的 for 循环”。作为提示，一种做法是维护一个索引向量（大小等于维度数量），然后在循环中手动递增它们（当某个索引达到最大值时，还需要执行类似“进位”的操作）。

However, if you get really stuck with this, you can alway use the fact that we're probably not going to ask you to deal with matrices of more than 6 dimensions (though we _will_ use 6 dimensions, for the im2col operation we discussed in class).

不过，如果你真的卡在这里，也可以利用这样一个事实：我们大概率不会要求你处理超过 6 维的矩阵（虽然我们 _会_ 使用 6 维数组，用于课堂上讨论过的 im2col 操作）。

#### The connection to setitem

#### 与 setitem 的联系

The setitem functionality, while seemingly quite different, is actually intimately related to `Compact()`.  `__setitem()__` is the Python function called when setting some elements of the object, i.e.,

setitem 功能看起来很不一样，但实际上与 `Compact()` 密切相关。`__setitem()__` 是在设置对象中某些元素时调用的 Python 函数，例如：

```python
A[::2,4:5,9] = 0 # or = some_other_array
```

How would you go about implementing this?  In the `__getitem()__` call above, you already implemented a method to take a subset of a matrix without copying (but just modifying strides).  But how would you actually go about _setting_ elements of this array?  In virtually all the other settings in this homework, we call `.compact()` before setting items in an output array, but in this case it doesn't work: calling `.compact()` would copy the subset array to some new memory, but the whole point of the `__setitem__()` call is that we want to modify existing memory.

你会如何实现它？在上面的 `__getitem()__` 调用中，你已经实现了一种无需复制即可取得矩阵子集的方法（只是修改步幅）。但你又该如何实际 _设置_ 这个数组中的元素呢？在本作业的几乎所有其他场景中，我们都会在设置输出数组元素之前调用 `.compact()`，但在这里这样做行不通：调用 `.compact()` 会把子集数组复制到一块新内存中，而 `__setitem__()` 调用的核心目的正是修改已有内存。

If you think about this for a while, you'll realize that the answer looks a lot like `.compact()` but in reverse.  If we want to assign a (itself already compact) right hand side matrix to a `__getitem()__` results, then we need to here like `shape` and `stride` be those fields of the _output_ matrix.  Then we could implement the setitem call as follows

如果你思考一会儿，就会发现答案看起来很像反向的 `.compact()`。如果我们想把一个（本身已经是 compact 的）右侧矩阵赋值给 `__getitem()__` 的结果，那么这里需要让 `shape` 和 `stride` 成为 _输出_ 矩阵的那些字段。然后我们就可以按如下方式实现 setitem 调用：

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // or "= val;"
```

Due to this similarity, if you implement your indexing strategy in a modular fashion, you'll be able to reuse it between the `Compact()` call and the `EwiseSetitem()` and `ScalarSetitem()` calls.

由于这种相似性，如果你以模块化方式实现索引策略，就能在 `Compact()` 调用以及 `EwiseSetitem()`、`ScalarSetitem()` 调用之间复用它。

**Note**: Don't forget to run make before executing the tests to rebuild your modified C++ code.

**注意**：执行测试之前不要忘记运行 `make`，以重新构建你修改过的 C++ 代码。


In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_compact_setitem"

## Part 3: CPU Backend - Elementwise and scalar operations

## 第 3 部分：CPU 后端 - 逐元素和标量操作

Implement the following functions in `ndarray_backend_cpu.cc`:

在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

You can look at the included
`EwiseAdd()` and `ScalarAdd()` functions (plus the invocations from `NDArray` in order to understand the required format of these functions.

你可以查看已经包含的 `EwiseAdd()` 和 `ScalarAdd()` 函数（以及 `NDArray` 中对它们的调用），以理解这些函数所需的格式。

Note that unlike the remaining functions mentioned here, we do not include function stubs for each of these functions.  This is because, while you can implement these naively just through implementing each function separately, though this will end up with a lot of duplicated code.  You're welcome to use e.g., C++ templates or macros to address this problem (but these would only be exposed internally, not to the external interface).

注意，与这里提到的其他函数不同，我们没有为这些函数逐一提供函数桩。这是因为，虽然你可以通过分别实现每个函数来进行朴素实现，但这样最终会产生大量重复代码。欢迎你使用例如 C++ 模板或宏来解决这个问题（不过这些只会在内部暴露，不会暴露给外部接口）。


In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_ops"

## Part 4: CPU Backend - Reductions

## 第 4 部分：CPU 后端 - 归约操作

Implement the following functions in `ndarray_backend_cpu.cc`:

在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

In general, the reduction functions `.max()` and `.sum()` in NDArray take the max or sum across a specified axis specified by the `axis` argument (or across the entire array when `axis=None`); note that we don't support axis being a set of axes, though this wouldn't be too hard to add if you desired (but it's not in the interface you should implement for the homework).

一般来说，NDArray 中的归约函数 `.max()` 和 `.sum()` 会沿着由 `axis` 参数指定的轴取最大值或求和（当 `axis=None` 时则在整个数组上进行归约）。注意，我们不支持 axis 是一组轴；如果你愿意，这并不难添加，但它不在本作业要求你实现的接口中。

Because summing over individual axes can be a bit tricky, even for compact arrays, these functions (in Python) in Python simplify things by permuting the last axis to the be the one reduced over (this is what the `reduce_view_out()` function in NDArray does), then compacting the array.  So for your `ReduceMax()` and `ReduceSum()` functions you implement in C++, you can assume that both the input and output arrays are contiguous in memory, and you want to just reduce over contiguous elements of size `reduce_size` as passed to the C++ functions. 

由于即使对于 compact 数组，沿单个轴求和也可能有点棘手，所以这些 Python 函数会通过把要归约的轴置换到最后一个轴来简化问题（这就是 NDArray 中 `reduce_view_out()` 函数所做的事情），然后再对数组进行 compact。因此，对于你在 C++ 中实现的 `ReduceMax()` 和 `ReduceSum()` 函数，可以假设输入数组和输出数组在内存中都是连续的，并且你只需要对传给 C++ 函数的大小为 `reduce_size` 的连续元素进行归约。


In [ ]:
!python3 -m pytest -v -k "reduce and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_reductions"

## Part 5: CPU Backend - Matrix multiplication

## 第 5 部分：CPU 后端 - 矩阵乘法

Implement the following functions in `ndarray_backend_cpu.cc`:

在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

The first implementation, `Matmul()` can use the naive three-nested-for-loops algorithm for matrix multiplication.  However, the `MatmulTiled()` performs the same matrix multiplication on memory laid out in tiled form, i.e., as a contiguous 4D array

第一个实现 `Matmul()` 可以使用朴素的三重嵌套 for 循环矩阵乘法算法。不过，`MatmulTiled()` 会在以 tiled 形式排布的内存上执行同样的矩阵乘法，也就是把内存看作一个连续的 4 维数组：

```c++
float[M/TILE][N/TILE][TILE][TILE];
```

Make sure to initialize the output matrix to zero before populating it with the correct values during matrix multiplication.

在矩阵乘法过程中填入正确值之前，务必先把输出矩阵初始化为零。

Note that the Python `__matmul__` code already does the conversion to tiled form when all sizes of the matrix multiplication are divisible by `TILE`, so your code just needs to implement the multiplication in this form.  In order to make the methods efficient, you will want to make use of (after you implement it), the `AlignedDot()` function, which will enable the compiler to efficiently make use of vector operations and proper caching.  The output matrix will also be in the tiled form above, and the Python backend will take care of the conversion to a normal 2D array.

注意，当矩阵乘法的所有尺寸都能被 `TILE` 整除时，Python 中的 `__matmul__` 代码已经会完成到 tiled 形式的转换，因此你的代码只需要实现这种形式下的乘法。为了让方法更高效，你会希望使用（在你实现之后）`AlignedDot()` 函数，它可以让编译器高效利用向量操作和合适的缓存。输出矩阵也会采用上面的 tiled 形式，而 Python 后端会负责将其转换回普通的二维数组。

Note that in order to get the most speedup possible from you tiled version, you may want to use the clang compiler with colab instead of gcc.  To do this, run the following command before building your code.

注意，为了让你的 tiled 版本获得尽可能大的加速，你可能希望在 Colab 中使用 clang 编译器而不是 gcc。为此，请在构建代码之前运行下面的命令。


In [ ]:
!python3 -m pytest -v -k "matmul and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_matmul"

In [ ]:
!export CXX=/usr/bin/clang++ && make

## Part 6: CUDA Backend - Compact and setitem

## 第 6 部分：CUDA 后端 - Compact 和 setitem

Implement the following functions in `ndarray_backend_cuda.cu`:

在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

For this portion, you'll implement the compact and setitem calls in the CUDA backend.  This is fairly similar to the C++ version, however, depending on how you implemented that function, there could also be some substantial differences.  We specifically want to highlight a few differences between the C++ and the CUDA implementations, however.

在这一部分，你将在 CUDA 后端中实现 compact 和 setitem 调用。这与 C++ 版本相当类似，不过根据你之前如何实现该函数，两者之间也可能存在一些显著差异。这里我们特别想强调 C++ 实现和 CUDA 实现之间的几个不同点。

First, as with the example functions implemented in the CUDA backend code, for all the functions above you will actually want to implement two functions: the basic functions listed above that you will call from Python, and the corresponding CUDA kernels that will actually perform the computation.  For the most part, we only provide the prototype for the "base" function in the `ndarray_backend_cuda.cu` file, and you will need to define and implement the kernel function yourself.  However, to see how these work, for the `Compact()` call we are providing you with the _complete_ `Compact()` call, and the function prototype for the `CompactKernel()` call.

首先，和 CUDA 后端代码中已经实现的示例函数一样，对于上面的所有函数，你实际上都需要实现两个函数：一个是上面列出的、会从 Python 调用的基础函数；另一个是实际执行计算的对应 CUDA kernel。大多数情况下，我们只在 `ndarray_backend_cuda.cu` 文件中提供“base”函数的原型，你需要自己定义并实现 kernel 函数。不过，为了展示这些函数如何工作，对于 `Compact()` 调用，我们提供了 _完整的_ `Compact()` 调用，以及 `CompactKernel()` 调用的函数原型。

One thing you may notice is the seemingly odd use of a `CudaVec` struct, which is a struct used to pass shape/stride parameters.  In the C++ version we used the STL `std::vector` variables to store these inputs (and the same is done in the base `Compact()` call, but CUDA kernels cannot operation on STL vectors, so something else is needed).  Furthermore, although we _could_ convert the vectors to normal CUDA arrays, this would be rather cumbersome, as we would need to call `cudaMalloc()`, pass the parameters as integer pointers, then free them after the calls.  Of course such memory management is needed for the actual underlying data in the array, but it seems like overkill to do it for just passing a variable-sized small tuple of shape/stride values.  The solution is to create a struct that has a "maximize" size for the number of dimensions an array can have, and then just store the actual shape/stride data in the first entries of these fields.  This is all done by the included `CudaVec` struct and `VecToCuda()` function, and you can just use these as provided for all the CUDA kernels that require passing shape/strides to the kernel itself.

你可能会注意到一个看起来有些奇怪的 `CudaVec` 结构体，它用于传递 shape/stride 参数。在 C++ 版本中，我们使用 STL 的 `std::vector` 变量来存储这些输入（基础 `Compact()` 调用中也是这样做的），但 CUDA kernel 不能操作 STL vector，所以需要别的方式。此外，虽然我们 _可以_ 把 vector 转换为普通的 CUDA 数组，但这会相当麻烦，因为我们需要调用 `cudaMalloc()`，把参数作为整数指针传入，然后在调用之后释放它们。当然，对于数组中真正的底层数据，这种内存管理是必要的；但如果只是为了传递一组长度可变的小型 shape/stride 元组，这样做就显得过于复杂了。解决方案是创建一个结构体，其中为数组可能拥有的维度数量设置一个“最大”大小，然后只把实际的 shape/stride 数据存储在这些字段的前几个条目中。所有这些都已经由提供的 `CudaVec` 结构体和 `VecToCuda()` 函数完成；对于所有需要把 shape/strides 传给 kernel 本身的 CUDA kernel，你都可以直接使用它们。

The other (more conceptual) big difference between the C++ and CUDA implementations of `Compact()` is that in C++ you will typically loop over the elements of the non-compact array sequentially, which allows you to perform some optimizations with respect to computing the corresponding indices between the compact and non-compact arrays.  In CUDA, you cannot do this, and will need to implement code that can directly map from an index in the compact array to one in the strided array.

C++ 和 CUDA 的 `Compact()` 实现之间另一个（更概念性的）重要区别在于，在 C++ 中你通常会顺序遍历非 compact 数组的元素，这允许你在计算 compact 数组和非 compact 数组之间对应索引时进行一些优化。在 CUDA 中，你不能这样做，而需要实现能够直接把 compact 数组中的索引映射到带步幅数组中对应索引的代码。

As before, we recommend you implement your code in such as way that it can easily be re-used between the `Compact()`, and `Setitem()` calls.  As a short note, remember that if you want to call a (separate, non-kernel) function from kernel code, you need to define it as a `__device__` function. In CUDA, `__global__` is used to define a function that runs on the GPU and is called from the CPU host, while `__device__` defines a function that runs and is called only on the GPU from other GPU function code

和之前一样，我们建议你以便于在 `Compact()` 和 `Setitem()` 调用之间复用的方式来实现代码。补充说明一下，如果你想从 kernel 代码中调用一个（单独的、非 kernel）函数，需要把它定义为 `__device__` 函数。在 CUDA 中，`__global__` 用于定义在 GPU 上运行并由 CPU host 调用的函数，而 `__device__` 定义的是只在 GPU 上运行、并且只能由其他 GPU 函数代码调用的函数。


In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_compact_setitem"

## Part 7: CUDA Backend - Elementwise and scalar operations

## 第 7 部分：CUDA 后端 - 逐元素和标量操作

Implement the following functions in `ndarray_backend_cuda.cu`:

在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

Again, we don't provide these function prototypes, and you're welcome to use C++ templates or macros to make this implementation more compact.  You will also want to uncomment the appropriate regions of the Pybind11 code once you've implemented each function.

同样，我们不会提供这些函数的原型，欢迎你使用 C++ 模板或宏来让这个实现更紧凑。每实现一个函数后，你还需要取消注释 Pybind11 代码中相应的区域。


In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_ops"

## Part 8: CUDA Backend - Reductions

## 第 8 部分：CUDA 后端 - 归约操作

Implement the following functions in `ndarray_backend_cuda.cu`:

在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

You can take a fairly simplistic approach here, and just use a separate CUDA thread for each individual reduction item: i.e., if there is a 100 x 20 array you are reducing over the second dimension, you could have 100 threads, each of which individually processed its own 20-dimensional array..  This is particularly inefficient for the `.max(axis=None)` calls, but we won't worry about this for the time being.  If you want a more industrial-grade implementation, you use a hierarchical mechanism that first aggregated across some smaller span, then had a secondary function that aggregated across _these_ reduced arrays, etc.  But this is not needed to pass the tests.

这里你可以采用相当简单的做法：为每个单独的归约项使用一个独立的 CUDA 线程。也就是说，如果有一个 100 x 20 的数组，你正在沿第二个维度进行归约，那么可以使用 100 个线程，每个线程分别处理自己的 20 维数组。对于 `.max(axis=None)` 这种调用，这尤其低效，但我们暂时不需要担心这一点。如果你想要更工业级的实现，可以使用分层机制：先在较小范围内聚合，然后用第二个函数在 _这些_ 已归约数组上继续聚合，依此类推。但通过测试并不需要这样做。


In [ ]:
!python3 -m pytest -v -k "reduce and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_reductions"

## Part 9: CUDA Backend - Matrix multiplication

## 第 9 部分：CUDA 后端 - 矩阵乘法

Implement the following functions in `ndarray_backend_cuda.cu`:

在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `Matmul()`

Finally, as your final exercise, you'll implement matrix multiplication on the GPU.  Your implementation here can roughly follow the presentation in class.  While you can pass the tests using fairly naive code here (i.e., you could just have a separate thread for each (i,j) location in the matrix, doing the matrix multiplication efficiently (to make it actually faster than a CPU version) requires cooperative fetching and the block shared memory register tiling covered in class.  Try to implement using these methods, and see how much faster you can get your code than the C++ (or numpy) backends.

最后，作为最终练习，你将在 GPU 上实现矩阵乘法。这里的实现可以大致遵循课堂上的讲解。虽然你可以用相当朴素的代码通过测试（也就是说，可以为矩阵中的每个 `(i,j)` 位置使用一个单独线程），但要高效地执行矩阵乘法（让它真正比 CPU 版本更快），就需要使用课堂上讲过的协作式读取以及 block 共享内存寄存器 tiling。尝试使用这些方法来实现，看看你的代码能比 C++（或 numpy）后端快多少。


In [ ]:
!python3 -m pytest -v -k "matmul and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_matmul"